# 3. Transformer Model — Sentence Embeddings + SVM Classifier
**Dataset:** CLINC150 — 151-class intent classification  
**Approach:** Custom preprocessing → `all-MiniLM-L6-v2` semantic encoder → LinearSVC  
**Expected Accuracy:** ~92–94%  
**Install:** `pip install sentence-transformers`

### Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output

# nltk.download('punkt'); nltk.download('stopwords')
# nltk.download('averaged_perceptron_tagger_eng'); nltk.download('wordnet')
print('All imports OK')

### Load Dataset

In [ ]:
test_dataset  = load_dataset('parquet', data_files={'test':       os.path.join('..', 'data', 'test-00000-of-00001.parquet')})
train_dataset = load_dataset('parquet', data_files={'train':      os.path.join('..', 'data', 'train-00000-of-00001.parquet')})
val_dataset   = load_dataset('parquet', data_files={'validation': os.path.join('..', 'data', 'validation-00000-of-00001.parquet')})

train_df = train_dataset['train'].to_pandas()
val_df   = val_dataset['validation'].to_pandas()
test_df  = test_dataset['test'].to_pandas()

NUM_CLASSES = train_df['intent'].nunique()
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Classes: {NUM_CLASSES}')
display(train_df.head(3))

### Preprocessing
Custom intent-aware preprocessing:
- **Keeps WH-words** (`what`, `where`, `who`, `why`, `how`, `when`, `do`) — critical for distinguishing intents like *weather* vs *directions*
- **Keeps `?` and `!`** — punctuation carries intent signal
- **POS-aware lemmatization** — reduces vocabulary without losing meaning
- **Removes remaining stopwords** — strips noise words like *the*, *a*, *is*

In [ ]:
def preprocess(sentences):
    base_stop_words  = set(stopwords.words('english'))
    words_to_keep    = {'what', 'where', 'who', 'why', 'how', 'when', 'do'}
    custom_stop_words = base_stop_words - words_to_keep
    allowed_symbols  = {'?', '!'}
    lemmatizer       = WordNetLemmatizer()

    result = []
    for sent in sentences:
        tokens   = word_tokenize(sent)
        pos_tags = nltk.pos_tag(tokens)
        cleaned  = []
        for token, pos in pos_tags:
            t = token.lower()
            if (t not in custom_stop_words) and (t.isalpha() or t in allowed_symbols):
                if t.isalpha():
                    if   pos.startswith('NN'): lemma = lemmatizer.lemmatize(t, 'n')
                    elif pos.startswith('VB'): lemma = lemmatizer.lemmatize(t, 'v')
                    elif pos.startswith('JJ'): lemma = lemmatizer.lemmatize(t, 'a')
                    else:                      lemma = lemmatizer.lemmatize(t)
                else:
                    lemma = t
                cleaned.append(lemma)
        result.append(' '.join(cleaned))
    return result


# Preview: raw vs preprocessed
samples = train_df['text'].tolist()[:5]
cleaned = preprocess(samples)
print(f'{"RAW":<45}  PREPROCESSED')
print('-' * 90)
for raw, pre in zip(samples, cleaned):
    print(f'{raw:<45}  {pre}')

### Encode with Sentence Transformer
`all-MiniLM-L6-v2` is an 80 MB BERT-based model optimised for semantic sentence similarity.  
Embeddings are **cached to disk** — encoding only runs once (~2–3 min on CPU).

In [ ]:
EMBED_CACHE = './clinc_embeddings_preprocessed.npz'
MODEL_NAME  = 'all-MiniLM-L6-v2'

if os.path.exists(EMBED_CACHE):
    print('Loading cached embeddings...')
    cache   = np.load(EMBED_CACHE)
    X_train = cache['X_train']
    X_val   = cache['X_val']
    X_test  = cache['X_test']
    print('Done.')
else:
    print('Preprocessing text...')
    train_texts = preprocess(train_df['text'].tolist())
    val_texts   = preprocess(val_df['text'].tolist())
    test_texts  = preprocess(test_df['text'].tolist())

    print(f'Encoding with {MODEL_NAME} (first run — ~2-3 min on CPU)...')
    encoder = SentenceTransformer(MODEL_NAME)
    X_train = encoder.encode(train_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    X_val   = encoder.encode(val_texts,   batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    X_test  = encoder.encode(test_texts,  batch_size=64, show_progress_bar=True, convert_to_numpy=True)

    np.savez_compressed(EMBED_CACHE, X_train=X_train, X_val=X_val, X_test=X_test)
    print(f'Saved to {EMBED_CACHE}')

y_train = train_df['intent'].tolist()
y_val   = val_df['intent'].tolist()
y_test  = test_df['intent'].tolist()

print(f'Embedding shape: {X_train.shape}')  # (N, 384)

### Train Classifier — LinearSVC
GridSearch over C. Training takes **< 30 seconds** on CPU.

In [ ]:
print('Running GridSearch over C...')
gs = GridSearchCV(
    LinearSVC(class_weight='balanced', max_iter=3000, random_state=42),
    {'C': [0.1, 1, 5, 10]},
    cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
gs.fit(X_train, y_train)
clf = gs.best_estimator_
print(f'Best C: {gs.best_params_["C"]}  |  CV Accuracy: {gs.best_score_*100:.2f}%')
joblib.dump(clf, './svm_on_minilm.pkl')
print('Classifier saved.')

### Evaluation

In [ ]:
y_val_pred  = clf.predict(X_val)
y_test_pred = clf.predict(X_test)

val_acc  = accuracy_score(y_val,  y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f'Validation Accuracy : {val_acc  * 100:.2f}%')
print(f'Test Accuracy       : {test_acc * 100:.2f}%')
print()
print('--- Test Classification Report ---')
print(classification_report(y_test, y_test_pred))

### Does Preprocessing Help?
We compare the preprocessed embeddings against raw text embeddings using the same classifier settings.
> **Note:** This cell re-encodes raw text — takes another ~2–3 min on first run.

In [ ]:
RAW_CACHE = './clinc_embeddings_raw.npz'

if os.path.exists(RAW_CACHE):
    cache_raw   = np.load(RAW_CACHE)
    X_train_raw = cache_raw['X_train']
    X_val_raw   = cache_raw['X_val']
    X_test_raw  = cache_raw['X_test']
    print('Loaded raw embeddings from cache.')
else:
    print('Encoding RAW text (no preprocessing)...')
    _enc        = SentenceTransformer(MODEL_NAME)
    X_train_raw = _enc.encode(train_df['text'].tolist(), batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    X_val_raw   = _enc.encode(val_df['text'].tolist(),   batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    X_test_raw  = _enc.encode(test_df['text'].tolist(),  batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    np.savez_compressed(RAW_CACHE, X_train=X_train_raw, X_val=X_val_raw, X_test=X_test_raw)

# Train same SVM on raw embeddings
gs_raw = GridSearchCV(
    LinearSVC(class_weight='balanced', max_iter=3000, random_state=42),
    {'C': [0.1, 1, 5, 10]}, cv=3, scoring='accuracy', n_jobs=-1
)
gs_raw.fit(X_train_raw, y_train)
clf_raw = gs_raw.best_estimator_

acc_raw_val  = accuracy_score(y_val,  clf_raw.predict(X_val_raw))
acc_raw_test = accuracy_score(y_test, clf_raw.predict(X_test_raw))
acc_pre_val  = accuracy_score(y_val,  clf.predict(X_val))
acc_pre_test = accuracy_score(y_test, clf.predict(X_test))

print(f'                    Val Acc   Test Acc')
print(f'Raw text          : {acc_raw_val*100:.2f}%    {acc_raw_test*100:.2f}%')
print(f'Preprocessed text : {acc_pre_val*100:.2f}%    {acc_pre_test*100:.2f}%')
diff = (acc_pre_test - acc_raw_test) * 100
direction = 'improvement' if diff >= 0 else 'drop'
print(f'\nPreprocessing {direction}: {abs(diff):.2f}%')
print()
print('Why preprocessing may HURT sentence transformers:')
print('  - The model was pretrained on natural full sentences')
print('  - Removing words like "is", "the" disrupts its learned patterns')
print('Why it may HELP:')
print('  - Keeping WH-words (what/where/why) preserves intent-critical signals')
print('  - Lemmatization reduces surface variation across training examples')

### Interactive Intent Predictor

In [ ]:
intent_mapping = {
    0: 'restaurant_reviews', 1: 'nutrition_info', 2: 'account_blocked', 3: 'oil_change_how', 4: 'time',
    5: 'weather', 6: 'redeem_rewards', 7: 'interest_rate', 8: 'gas_type', 9: 'accept_reservations',
    10: 'smart_home', 11: 'user_name', 12: 'report_lost_card', 13: 'repeat', 14: 'whisper_mode',
    15: 'what_are_your_hobbies', 16: 'order', 17: 'jump_start', 18: 'schedule_meeting', 19: 'meeting_schedule',
    20: 'freeze_account', 21: 'what_song', 22: 'meaning_of_life', 23: 'restaurant_reservation', 24: 'traffic',
    25: 'make_call', 26: 'text', 27: 'bill_balance', 28: 'improve_credit_score', 29: 'change_language',
    30: 'no', 31: 'measurement_conversion', 32: 'timer', 33: 'flip_coin', 34: 'do_you_have_pets',
    35: 'balance', 36: 'tell_joke', 37: 'last_maintenance', 38: 'exchange_rate', 39: 'uber',
    40: 'car_rental', 41: 'credit_limit', 42: 'oos', 43: 'shopping_list', 44: 'expiration_date',
    45: 'routing', 46: 'meal_suggestion', 47: 'tire_change', 48: 'todo_list', 49: 'card_declined',
    50: 'rewards_balance', 51: 'change_accent', 52: 'vaccines', 53: 'reminder_update', 54: 'food_last',
    55: 'change_ai_name', 56: 'bill_due', 57: 'who_do_you_work_for', 58: 'share_location', 59: 'international_visa',
    60: 'calendar', 61: 'translate', 62: 'carry_on', 63: 'book_flight', 64: 'insurance_change',
    65: 'todo_list_update', 66: 'timezone', 67: 'cancel_reservation', 68: 'transactions', 69: 'credit_score',
    70: 'report_fraud', 71: 'spending_history', 72: 'directions', 73: 'spelling', 74: 'insurance',
    75: 'what_is_your_name', 76: 'reminder', 77: 'where_are_you_from', 78: 'distance', 79: 'payday',
    80: 'flight_status', 81: 'find_phone', 82: 'greeting', 83: 'alarm', 84: 'order_status',
    85: 'confirm_reservation', 86: 'cook_time', 87: 'damaged_card', 88: 'reset_settings', 89: 'pin_change',
    90: 'replacement_card_duration', 91: 'new_card', 92: 'roll_dice', 93: 'income', 94: 'taxes',
    95: 'date', 96: 'who_made_you', 97: 'pto_request', 98: 'tire_pressure', 99: 'how_old_are_you',
    100: 'rollover_401k', 101: 'pto_request_status', 102: 'how_busy', 103: 'application_status', 104: 'recipe',
    105: 'calendar_update', 106: 'play_music', 107: 'yes', 108: 'direct_deposit', 109: 'credit_limit_change',
    110: 'gas', 111: 'pay_bill', 112: 'ingredients_list', 113: 'lost_luggage', 114: 'goodbye',
    115: 'what_can_i_ask_you', 116: 'book_hotel', 117: 'are_you_a_bot', 118: 'next_song', 119: 'change_speed',
    120: 'plug_type', 121: 'maybe', 122: 'w2', 123: 'oil_change_when', 124: 'thank_you', 125: 'shopping_list_update',
    126: 'pto_balance', 127: 'order_checks', 128: 'travel_alert', 129: 'fun_fact', 130: 'sync_device',
    131: 'schedule_maintenance', 132: 'apr', 133: 'transfer', 134: 'ingredient_substitution', 135: 'calories',
    136: 'current_location', 137: 'international_fees', 138: 'calculator', 139: 'definition', 140: 'next_holiday',
    141: 'update_playlist', 142: 'mpg', 143: 'min_payment', 144: 'change_user_name', 145: 'restaurant_suggestion',
    146: 'travel_notification', 147: 'cancel', 148: 'pto_used', 149: 'travel_suggestion', 150: 'change_volume'
}

# Load artifacts
_encoder = SentenceTransformer('all-MiniLM-L6-v2')
_clf     = joblib.load('./svm_on_minilm.pkl')

def predict_intent(text: str):
    cleaned = preprocess([text])[0]
    vec     = _encoder.encode([cleaned], convert_to_numpy=True)
    pred_id = _clf.predict(vec)[0]
    return pred_id, intent_mapping.get(pred_id, str(pred_id)), cleaned

# ── Widgets ──────────────────────────────────────────────────────────────────
text_input = widgets.Text(
    value='',
    placeholder='e.g. what is the weather tomorrow?',
    description='Input:',
    layout=widgets.Layout(width='70%')
)
predict_btn = widgets.Button(
    description='Predict Intent',
    button_style='success',
    layout=widgets.Layout(width='150px')
)
clear_btn = widgets.Button(
    description='Clear',
    button_style='warning',
    layout=widgets.Layout(width='80px')
)
output_area = widgets.Output(
    layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='10px 0')
)

def on_predict(b):
    with output_area:
        clear_output()
        user_text = text_input.value.strip()
        if not user_text:
            print('⚠️  Please type a sentence first.')
            return
        pred_id, intent_label, preprocessed = predict_intent(user_text)
        print(f'Input text       : {user_text}')
        print(f'After preprocess : {preprocessed}')
        print(f'Predicted intent : [{intent_label.upper()}]  (class id: {pred_id})')

def on_clear(b):
    text_input.value = ''
    with output_area:
        clear_output()

predict_btn.on_click(on_predict)
clear_btn.on_click(on_clear)

display(
    widgets.HBox([text_input, predict_btn, clear_btn]),
    output_area
)